# Prior-Study Viability

Goal: decide whether incorporating a patient's previous CXR study into the CaMCheX model is worth the engineering effort.

We check:
1. **Coverage** — what fraction of studies have a recorded prior?
2. **History depth** — how many priors per patient (longitudinal richness)?
3. **Time gap** — days between current and prior study (is the prior still informative?)
4. **Label dynamics** — do labels actually *change* between visits? (If priors == current, the model just memorizes the prior.)
5. **Per-class priors-helpfulness proxy** — conditional label rates given the prior had the same finding.
6. **Split safety** — are prior studies always in the same split as the current (no leakage)?
7. **View availability** — does the prior have a frontal view we can encode?
8. **Prior chain depth** — how many consecutive priors can we follow? (for recurrent / multi-prior models)
9. **Time gap × label change interaction** — are shorter-gap priors more informative?
10. **Per-class transition heatmap** — which findings flip most between prior and current?
11. **Summary** — consolidated verdict table.

In [ ]:
import os
os.environ['CURRENT_NOTEBOOK_NAME'] = 'prior-study-viability'
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils import *

In [ ]:
root_dir, data_dir = get_notebook_paths()
DATA_CAMCHEX = data_dir / "data-camchex"
MIMIC_CXR_JPG = data_dir / "MIMIC-CXR-JPG"

print(Path.cwd())
print(root_dir)
print(data_dir)

In [ ]:
CLASSES = [
    'Atelectasis','Calcification of the Aorta','Cardiomegaly','Consolidation','Edema',
    'Emphysema','Enlarged Cardiomediastinum','Fibrosis','Fracture','Hernia',
    'Infiltration','Lung Lesion','Lung Opacity','Mass','No Finding','Nodule',
    'Pleural Effusion','Pleural Other','Pleural Thickening','Pneumomediastinum',
    'Pneumonia','Pneumoperitoneum','Pneumothorax','Subcutaneous Emphysema',
    'Support Devices','Tortuous Aorta'
]

frames = []
for split in ["train", "development", "test"]:
    p = DATA_CAMCHEX / f"02_{split}.csv"
    df = pd.read_csv(p, low_memory=False)
    df["split"] = split
    frames.append(df)
df_row = pd.concat(frames, ignore_index=True)

# Collapse to study-level (one row per study)
df = df_row.drop_duplicates("study_id").copy()
df["StudyDateTime"] = pd.to_datetime(df["StudyDateTime"], errors="coerce")
df["has_prior"] = df["PreviousStudy"].notna()
print("row-level:", len(df_row), "study-level:", len(df))
df.head(3)

## 1. Coverage — fraction of studies with a prior

In [ ]:
cov = df.groupby("split")["has_prior"].agg(["sum", "count"])
cov["ratio"] = cov["sum"] / cov["count"]
cov.loc["ALL"] = [df["has_prior"].sum(), len(df), df["has_prior"].mean()]
print(cov)

fig, ax = plt.subplots(figsize=(6, 3.5))
cov_plot = cov.drop("ALL").reset_index()
sns.barplot(data=cov_plot, x="split", y="ratio", ax=ax)
for i, r in enumerate(cov_plot["ratio"]):
    ax.text(i, r + 0.01, f"{r:.1%}", ha="center")
ax.set_ylim(0, 1)
ax.set_ylabel("Fraction of studies with prior")
ax.set_title("Prior-study coverage by split")
plt.tight_layout(); plt.show()

## 2. History depth — studies per patient

In [ ]:
studies_per_patient = df.groupby("subject_id")["study_id"].nunique()
print(studies_per_patient.describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
spp_clip = studies_per_patient.clip(upper=16)
sns.histplot(spp_clip, bins=range(1, 17), discrete=True, ax=axes[0])
axes[0].set_xlabel("# studies per patient (clipped at 16)")
axes[0].set_ylabel("# patients")
axes[0].set_title("Distribution of studies per patient")

buckets = pd.cut(studies_per_patient, bins=[0, 1, 2, 3, 5, 10, 1e9],
                 labels=["1", "2", "3", "4-5", "6-10", "11+"])
buckets.value_counts(sort=False).plot(kind="bar", ax=axes[1])
axes[1].set_ylabel("# patients")
axes[1].set_title("Bucketed history depth")
for p in axes[1].patches:
    axes[1].annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                     ha="center", va="bottom")
plt.tight_layout(); plt.show()

## 3. Time gap to prior
If the prior was years ago, it's probably stale; if it was minutes ago (same admission), it's nearly identical.

In [ ]:
date_lookup = df.set_index("study_id")["StudyDateTime"].to_dict()

def _lookup_prev_date(prev_id):
    """Resolve a PreviousStudy ID to its StudyDateTime."""
    try:
        return date_lookup.get(int(prev_id))
    except (ValueError, TypeError):
        return None

df["prev_date"] = df["PreviousStudy"].map(_lookup_prev_date)
df["days_since_prev"] = (df["StudyDateTime"] - df["prev_date"]).dt.total_seconds() / 86400

gaps = df["days_since_prev"].dropna()
print("gaps with both dates known:", len(gaps), "of", df["has_prior"].sum(), "priors")
print(gaps.describe(percentiles=[.1, .25, .5, .75, .9, .99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(gaps.clip(0, 365*3), bins=60, ax=axes[0])
axes[0].set_xlabel("Days since prior (clipped at 3 years)")
axes[0].set_title("Time-gap distribution (linear)")

sns.histplot(np.log1p(gaps[gaps >= 0]), bins=60, ax=axes[1])
axes[1].set_xlabel("log(1 + days since prior)")
axes[1].set_title("Time-gap distribution (log)")
plt.tight_layout(); plt.show()

# Bucket gaps into clinically meaningful windows
buckets = pd.cut(gaps, bins=[-0.5, 1, 7, 30, 180, 365, 365*3, np.inf],
                 labels=["≤1d", "2-7d", "8-30d", "1-6mo", "6-12mo", "1-3y", ">3y"])
fig, ax = plt.subplots(figsize=(7, 3.5))
buckets.value_counts(sort=False).plot(kind="bar", ax=ax)
ax.set_title("Time gap buckets")
ax.set_ylabel("# studies")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout(); plt.show()

## 4. Label dynamics — do findings change between visits?
If labels never change between current and prior, the prior is just a leak / shortcut. We want some change, but not total chaos.

In [ ]:
label_lookup = df.set_index("study_id")[CLASSES].astype(float)

paired = df.dropna(subset=["PreviousStudy"]).copy()
paired["PreviousStudy_int"] = paired["PreviousStudy"].astype("Int64")
paired = paired[paired["PreviousStudy_int"].isin(label_lookup.index)]

cur = label_lookup.loc[paired["study_id"].values].fillna(0).values
prv = label_lookup.loc[paired["PreviousStudy_int"].astype(int).values].fillna(0).values
cur_b = (cur > 0).astype(int)
prv_b = (prv > 0).astype(int)

agree = (cur_b == prv_b).mean(axis=0)
new_pos = ((cur_b == 1) & (prv_b == 0)).mean(axis=0)
lost_pos = ((cur_b == 0) & (prv_b == 1)).mean(axis=0)
prev_pos = prv_b.mean(axis=0)
cur_pos = cur_b.mean(axis=0)

dyn = pd.DataFrame({
    "class": CLASSES,
    "prev_prevalence": prev_pos,
    "cur_prevalence": cur_pos,
    "agreement": agree,
    "new_positive": new_pos,
    "lost_positive": lost_pos,
}).sort_values("cur_prevalence", ascending=False)
dyn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
order = dyn.sort_values("agreement").reset_index(drop=True)
axes[0].barh(order["class"], order["agreement"], color="steelblue")
axes[0].axvline(0.9, color="red", ls="--", lw=1, label="0.90 (very static)")
axes[0].axvline(0.5, color="orange", ls="--", lw=1, label="0.50 (coin flip)")
axes[0].set_xlabel("P(current label = prior label)")
axes[0].set_title("How often does the label stay the same?")
axes[0].legend()

flip = order[["class", "new_positive", "lost_positive"]].set_index("class")
flip.plot(kind="barh", stacked=True, ax=axes[1], color=["#2ca02c", "#d62728"])
axes[1].set_xlabel("Fraction of pairs")
axes[1].set_title("Label transitions (new positive vs. lost positive)")
plt.tight_layout(); plt.show()

## 5. Is the prior actually informative?
P(current label = 1 | prior label = 1) vs. P(current label = 1 | prior label = 0).
A large gap means the prior carries real signal for that class.

In [ ]:
eps = 1e-9
p_cur_given_prv1 = (cur_b * prv_b).sum(0) / (prv_b.sum(0) + eps)
p_cur_given_prv0 = (cur_b * (1 - prv_b)).sum(0) / ((1 - prv_b).sum(0) + eps)
lift = p_cur_given_prv1 - p_cur_given_prv0

info = pd.DataFrame({
    "class": CLASSES,
    "P(cur=1 | prv=1)": p_cur_given_prv1,
    "P(cur=1 | prv=0)": p_cur_given_prv0,
    "lift": lift,
    "base_rate": cur_pos,
}).sort_values("lift", ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
y = np.arange(len(info))
ax.barh(y, info["P(cur=1 | prv=1)"], color="#2ca02c", label="prv=1")
ax.barh(y, info["P(cur=1 | prv=0)"], color="#d62728", alpha=0.6, label="prv=0")
ax.set_yticks(y); ax.set_yticklabels(info["class"])
ax.invert_yaxis()
ax.set_xlabel("P(current = 1)")
ax.set_title("Prior label informativeness per class")
ax.legend()
plt.tight_layout(); plt.show()
info

## 6. Split safety — is the prior always in the same split?
If a `test` sample's prior lives in `train`, we'd leak training images into eval. The dataset should keep all of a patient's studies in one split.

In [ ]:
split_lookup = df.set_index("study_id")["split"].to_dict()
paired = df.dropna(subset=["PreviousStudy"]).copy()
paired["prev_split"] = paired["PreviousStudy"].map(
    lambda x: split_lookup.get(int(x)) if pd.notna(x) else None
)
paired = paired.dropna(subset=["prev_split"])
leak = pd.crosstab(paired["split"], paired["prev_split"])
print(leak)
leak_norm = leak.div(leak.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(leak_norm, annot=True, fmt=".2%", cmap="Blues", ax=ax)
ax.set_title("P(prior split | current split)")
plt.tight_layout(); plt.show()

## 7. View availability — does the prior have a frontal view we can encode?
If most priors are single-view AP/PA only, that's fine — means cheap to encode.

In [ ]:
views_per_study = df_row.groupby("study_id")["ViewPosition"].apply(
    lambda s: set(str(v).upper() for v in s.dropna())
)
n_views = views_per_study.apply(len)
print(n_views.describe())

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.histplot(n_views.clip(upper=6), bins=range(1, 8), discrete=True, ax=ax)
ax.set_xlabel("# distinct view positions per study (clipped at 6)")
ax.set_ylabel("# studies")
ax.set_title("Views per study")
plt.tight_layout(); plt.show()

from collections import Counter
view_counts = Counter(v for s in views_per_study for v in s)
print(pd.Series(view_counts).sort_values(ascending=False).head(10))

In [ ]:
# View-type breakdown for prior studies specifically
prior_ids = df.dropna(subset=["PreviousStudy"])["PreviousStudy"].astype(int).unique()
prior_views = views_per_study.reindex(prior_ids).dropna()

FRONTAL = {"AP", "PA"}
has_frontal = prior_views.apply(lambda s: bool(s & FRONTAL))
has_lateral = prior_views.apply(lambda s: bool(s & {"LATERAL", "LL"}))

view_breakdown = pd.DataFrame({
    "Frontal only": [( has_frontal & ~has_lateral).sum()],
    "Lateral only": [(~has_frontal &  has_lateral).sum()],
    "Both":         [( has_frontal &  has_lateral).sum()],
    "Neither":      [(~has_frontal & ~has_lateral).sum()],
}).T
view_breakdown.columns = ["count"]
view_breakdown["pct"] = view_breakdown["count"] / view_breakdown["count"].sum() * 100
print(view_breakdown)

fig, ax = plt.subplots(figsize=(6, 3.5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
wedges, texts, autotexts = ax.pie(
    view_breakdown["count"], labels=view_breakdown.index,
    autopct="%1.1f%%", colors=colors, startangle=90,
    textprops={"fontsize": 10}
)
ax.set_title("Prior studies: view availability")
plt.tight_layout(); plt.show()

## 8. Prior chain depth — how many consecutive priors can we follow?
For recurrent or multi-prior models, it matters whether we can chain
`current → prior → prior-of-prior → …` If most chains are length 1,
only the immediate prior is usable.

In [ ]:
prior_map = df.dropna(subset=["PreviousStudy"]).set_index("study_id")["PreviousStudy"].to_dict()

def chain_length(study_id, prior_map, max_depth=50):
    """Walk the prior chain from study_id and return its length."""
    depth = 0
    current = study_id
    visited = set()
    while current in prior_map and depth < max_depth:
        if current in visited:  # cycle guard
            break
        visited.add(current)
        try:
            current = int(prior_map[current])
        except (ValueError, TypeError):
            break
        depth += 1
    return depth

chain_depths = df["study_id"].apply(lambda sid: chain_length(sid, prior_map))
print("Chain depth stats (0 = no prior):")
print(chain_depths.describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
chain_clip = chain_depths.clip(upper=10)
sns.histplot(chain_clip, bins=range(0, 12), discrete=True, ax=axes[0])
axes[0].set_xlabel("Prior chain depth (clipped at 10)")
axes[0].set_ylabel("# studies")
axes[0].set_title("Distribution of prior chain depth")

# Bucketed
depth_buckets = pd.cut(chain_depths, bins=[-0.5, 0.5, 1.5, 2.5, 5.5, 100],
                       labels=["0 (none)", "1", "2", "3-5", "6+"])
depth_counts = depth_buckets.value_counts(sort=False)
depth_counts.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_ylabel("# studies")
axes[1].set_title("Bucketed prior chain depth")
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

pct_multi = (chain_depths >= 2).mean() * 100
print(f"\n{pct_multi:.1f}% of studies have chain depth ≥ 2 (could benefit from multi-prior)")

## 9. Time gap × label change interaction
Do shorter-gap priors carry more predictive signal? We bucket the gap
and compute per-bucket Hamming distance (fraction of labels that flip).

In [ ]:
# Build paired data with gap info
paired_gap = df.dropna(subset=["PreviousStudy", "days_since_prev"]).copy()
paired_gap["PreviousStudy_int"] = paired_gap["PreviousStudy"].astype("Int64")
paired_gap = paired_gap[paired_gap["PreviousStudy_int"].isin(label_lookup.index)]
paired_gap = paired_gap[paired_gap["days_since_prev"] >= 0]

cur_g = label_lookup.loc[paired_gap["study_id"].values].fillna(0).values
prv_g = label_lookup.loc[paired_gap["PreviousStudy_int"].astype(int).values].fillna(0).values
cur_gb = (cur_g > 0).astype(int)
prv_gb = (prv_g > 0).astype(int)

# Hamming distance = fraction of labels that differ
hamming = (cur_gb != prv_gb).mean(axis=1)
paired_gap = paired_gap.assign(hamming=hamming)

# Bucket the time gaps
paired_gap["gap_bucket"] = pd.cut(
    paired_gap["days_since_prev"],
    bins=[-0.5, 1, 7, 30, 180, 365, 365*3, np.inf],
    labels=["≤1d", "2-7d", "8-30d", "1-6mo", "6-12mo", "1-3y", ">3y"]
)

gap_stats = paired_gap.groupby("gap_bucket", observed=True)["hamming"].agg(["mean", "std", "count"])
print(gap_stats)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Bar: mean Hamming distance per gap bucket
axes[0].bar(range(len(gap_stats)), gap_stats["mean"], yerr=gap_stats["std"],
            capsize=3, color="steelblue", alpha=0.85)
axes[0].set_xticks(range(len(gap_stats)))
axes[0].set_xticklabels(gap_stats.index, rotation=0)
axes[0].set_ylabel("Mean Hamming distance (frac. labels changed)")
axes[0].set_title("Label change rate by time gap")
axes[0].set_ylim(0, None)

# Scatter: days_since_prev vs hamming (subsample for readability)
sample = paired_gap.sample(min(5000, len(paired_gap)), random_state=42)
axes[1].scatter(sample["days_since_prev"], sample["hamming"],
                alpha=0.15, s=8, color="steelblue")
# Trend line via rolling median
sorted_data = paired_gap.sort_values("days_since_prev")
window = max(len(sorted_data) // 50, 100)
rolling_med = sorted_data["hamming"].rolling(window, center=True, min_periods=50).median()
axes[1].plot(sorted_data["days_since_prev"].values, rolling_med.values,
             color="red", lw=2, label=f"rolling median (w={window})")
axes[1].set_xlabel("Days since prior")
axes[1].set_ylabel("Hamming distance")
axes[1].set_title("Label change vs. time gap (scatter + trend)")
axes[1].set_xscale("symlog", linthresh=1)
axes[1].legend()

plt.tight_layout(); plt.show()

## 10. Per-class transition heatmap
For the top-N most dynamic classes, show a 4-cell confusion matrix:
`(prior=0,cur=0), (prior=0,cur=1), (prior=1,cur=0), (prior=1,cur=1)`.
This reveals which findings are "sticky" (high 1→1) vs. transient (high 1→0).

In [ ]:
# Compute the 4-cell transition fractions for each class
n_pairs = cur_b.shape[0]
transitions = pd.DataFrame({
    "class": CLASSES,
    "0→0": ((cur_b == 0) & (prv_b == 0)).sum(0) / n_pairs,
    "0→1": ((cur_b == 1) & (prv_b == 0)).sum(0) / n_pairs,  # new positive
    "1→0": ((cur_b == 0) & (prv_b == 1)).sum(0) / n_pairs,  # lost positive
    "1→1": ((cur_b == 1) & (prv_b == 1)).sum(0) / n_pairs,  # persistent
}).set_index("class")

# Sort by total change rate (0→1 + 1→0)
transitions["change_rate"] = transitions["0→1"] + transitions["1→0"]
transitions = transitions.sort_values("change_rate", ascending=False)

# Show top 15 most dynamic classes
top_n = min(15, len(transitions))
top = transitions.head(top_n)

fig, ax = plt.subplots(figsize=(8, 7))
hm_data = top[["0→0", "0→1", "1→0", "1→1"]]
sns.heatmap(hm_data, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax,
            linewidths=0.5, cbar_kws={"label": "Fraction of paired studies"})
ax.set_title(f"Label transitions: prior→current (top {top_n} by change rate)")
ax.set_ylabel("")
plt.tight_layout(); plt.show()

# Persistence rate: P(cur=1 | prv=1) for each class
persist = transitions["1→1"] / (transitions["1→0"] + transitions["1→1"] + 1e-9)
persist_df = persist.reset_index()
persist_df.columns = ["class", "persistence_rate"]
persist_df = persist_df.sort_values("persistence_rate", ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
colors = ["#d62728" if r < 0.5 else "#2ca02c" if r > 0.75 else "#ff7f0e"
          for r in persist_df["persistence_rate"]]
ax.barh(persist_df["class"], persist_df["persistence_rate"], color=colors)
ax.axvline(0.5, color="gray", ls="--", lw=1, alpha=0.7)
ax.axvline(0.75, color="gray", ls=":", lw=1, alpha=0.7)
ax.set_xlabel("P(cur=1 | prior=1)  — persistence rate")
ax.set_title("How sticky is each finding?")
ax.set_xlim(0, 1.05)
for i, (_, row) in enumerate(persist_df.iterrows()):
    ax.text(row["persistence_rate"] + 0.01, i, f"{row['persistence_rate']:.2f}",
            va="center", fontsize=8)
plt.tight_layout(); plt.show()

## 11. Summary — consolidated verdict
Collect all key metrics into one table for quick reference.

In [ ]:
overall_coverage = df["has_prior"].mean()
median_gap = gaps[gaps >= 0].median()
mean_agreement = agree.mean()
mean_lift = lift.mean()
max_lift_class = CLASSES[int(np.argmax(lift))]
max_lift_val = lift.max()

# Leak rate: fraction of cross-split prior references
leak_diag = np.diag(leak_norm.values) if hasattr(leak_norm, 'values') else [1.0]
min_diag = min(leak_diag)

# Chain depth
pct_chain2 = (chain_depths >= 2).mean() * 100

# Frontal availability for priors
pct_frontal = has_frontal.mean() * 100 if len(has_frontal) > 0 else 0

summary = pd.DataFrame([
    ["Overall prior coverage", f"{overall_coverage:.1%}", "≥50%", "✅" if overall_coverage >= 0.5 else "⚠️"],
    ["Median studies/patient", f"{studies_per_patient.median():.0f}", "≥2", "✅" if studies_per_patient.median() >= 2 else "⚠️"],
    ["Median time gap (days)", f"{median_gap:.0f}", "≤180", "✅" if median_gap <= 180 else "⚠️"],
    ["Mean label agreement", f"{mean_agreement:.3f}", "0.7–0.95", "✅" if 0.7 <= mean_agreement <= 0.95 else "⚠️"],
    ["Mean prior-label lift", f"{mean_lift:.3f}", ">0.05", "✅" if mean_lift > 0.05 else "⚠️"],
    ["Top lift class", f"{max_lift_class} ({max_lift_val:.3f})", "-", "-"],
    ["Min split-safety (diagonal)", f"{min_diag:.2%}", "≥99%", "✅" if min_diag >= 0.99 else "⚠️"],
    ["Prior frontal availability", f"{pct_frontal:.1f}%", "≥80%", "✅" if pct_frontal >= 80 else "⚠️"],
    ["Studies with chain ≥ 2", f"{pct_chain2:.1f}%", ">10%", "✅" if pct_chain2 > 10 else "⚠️"],
], columns=["Metric", "Value", "Target", "Status"])

display(summary.style.hide(axis='index').set_caption("Prior-study viability verdict"))

## Table 8 (recalculated): Availability of prior studies at study level

Recalculate the prior-study availability table to verify the numbers in the report.

In [ ]:
# ── Table 8 recalculation: Prior-study availability at study level ──
# df is already study-level (one row per study_id) with has_prior column.

n_with = df["has_prior"].sum()
n_without = (~df["has_prior"]).sum()
n_total = len(df)

table8 = pd.DataFrame({
    "Prior-study status": ["With prior study", "Without prior study", "Total"],
    "Count": [int(n_with), int(n_without), int(n_total)],
    "Percentage": [f"{n_with / n_total:.2%}", f"{n_without / n_total:.2%}", "100.00%"],
}).set_index("Prior-study status")

print("Table 8: Availability of prior studies at study level")
display(table8)

# Also show per-split breakdown
print("Per-split breakdown:")
per_split = df.groupby("split")["has_prior"].agg(
    with_prior="sum", total="count"
)
per_split["without_prior"] = per_split["total"] - per_split["with_prior"]
per_split["pct_with"] = (per_split["with_prior"] / per_split["total"]).map("{:.2%}".format)
per_split = per_split[["with_prior", "without_prior", "total", "pct_with"]].astype(
    {"with_prior": int, "without_prior": int, "total": int}
)
display(per_split)


## Verdict checklist

Looking at the plots above, the prior-study direction is **viable** when:
- **Coverage** ≳ 50%. If most studies have no prior, the model has to learn two modes (with/without prior) on top of the harder task — still doable, but the ablation gain is muted.
- **History depth** has a heavy tail. Many patients with 3+ visits = clear case for extending beyond just-prior-one.
- **Time gaps** are not all >1y. Sub-month gaps are the cleanest case; if median is 1–6mo we're in good shape.
- **Label agreement** is high but not 1.0. ~0.7–0.95 per class is the sweet spot — priors are predictive but not identical, so the model can't just copy.
- **Lift** ( P(cur|prv=1) − P(cur|prv=0) ) is large for chronic findings (Cardiomegaly, Calcification of the Aorta, Tortuous Aorta, Support Devices). That's the prior's value proposition.
- **Split safety**: diagonal of the leak heatmap should be ~100%. If not, fix the split before training.
- **Chain depth**: if many studies have depth ≥ 2, consider multi-prior architectures (e.g., recurrent encoders, attention over a prior sequence).
- **Time gap × label change**: if Hamming distance increases with gap, the model should learn to weight recent priors more heavily (temporal attention/decay).